# 01 — Korpus & Annotation

Phase 1 (Korpus + Bias-Notiz) und Phase 2 (κ + drei Edge Cases) leben in diesem Notebook. Pipeline → `02_extract.ipynb`, Eval → `03_eval.ipynb`, Frontier → `04_frontier_compare.ipynb`.

## Run-Header

| Feld | Wert |
|---|---|
| Datum (Phase 1) | _YYYY-MM-DD_ |
| Datum (Phase 2) | _YYYY-MM-DD_ |
| Korpus-Datei | `daten/eigener_korpus.jsonl` |
| Anzahl Anzeigen im Korpus | _ |
| Genutzte Suchanfragen (Phase 1) | _ |
| Pair-Partner:in (Phase 2) | _ |
| 12 gemeinsame Anzeigen-IDs | _ |

## Phase 1 — Korpus aufbauen + inspizieren

### Block 1.1 — Korpus von der Bundesagentur-API ziehen

Die [Jobsuche-API der Bundesagentur](https://github.com/bundesAPI/jobsuche-api) liefert Suchergebnisse + Detail-Beschreibungen. Header `X-API-Key: jobboerse-jobsuche` ist öffentlich dokumentiert.

Zwei Endpoints:
- `GET /pc/v4/jobs?was=…&page=…&size=…` — paginierte Suche, liefert Liste von Stellenangeboten mit `refnr`
- `GET /pc/v4/jobdetails/{hashId}` — Detail-Beschreibung pro Anzeige. `hashId` = base64(refnr) ohne Padding

Pro Anzeige speichern wir mind. `refnr`, `titel`, `firma`, `text` plus die Strukturfelder, die die API ohnehin mitschickt (für Block 1.2).

In [ ]:
import base64
import json
import time
from pathlib import Path

import requests

API_BASE = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service"
HEADERS = {"X-API-Key": "jobboerse-jobsuche"}

KORPUS_PATH = Path("../daten/eigener_korpus.jsonl")
KORPUS_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def search(was: str, wo: str | None = None, size: int = 50, page: int = 1) -> dict:
    params = {"was": was, "page": page, "size": size}
    if wo:
        params["wo"] = wo
    r = requests.get(f"{API_BASE}/pc/v4/jobs", params=params, headers=HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()


def detail(refnr: str) -> dict:
    # API erwartet base64-encoded refnr ohne Padding
    hash_id = base64.b64encode(refnr.encode("utf-8")).decode("ascii").rstrip("=")
    r = requests.get(f"{API_BASE}/pc/v4/jobdetails/{hash_id}", headers=HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()


def extrahiere_treffer(such_response: dict, suchbegriff: str) -> list[dict]:
    """Holt aus einer Such-Response die Mindestfelder + behält Strukturfelder fürs Bias-Audit."""
    out = []
    for st in such_response.get("stellenangebote", []):
        out.append({
            "refnr": st.get("refnr"),
            "titel": st.get("titel") or st.get("beruf"),
            "firma": st.get("arbeitgeber"),
            "ort": (st.get("arbeitsort") or {}).get("ort"),
            "plz": (st.get("arbeitsort") or {}).get("plz"),
            "region": (st.get("arbeitsort") or {}).get("region"),
            "eintrittsdatum": st.get("eintrittsdatum"),
            "aktuelleVeroeffentlichungsdatum": st.get("aktuelleVeroeffentlichungsdatum"),
            "externeUrl": st.get("externeUrl"),
            "hashId": st.get("hashId"),
            "_suchbegriff": suchbegriff,
        })
    return out


def reichere_mit_text_an(eintrag: dict, sleep: float = 0.4) -> dict:
    """Holt die Detail-Beschreibung und hängt sie als `text` an den Eintrag."""
    try:
        d = detail(eintrag["refnr"])
    except requests.HTTPError as e:
        eintrag["text"] = ""
        eintrag["_detail_fehler"] = str(e)
        return eintrag
    eintrag["text"] = d.get("stellenbeschreibung") or ""
    # zusätzliche Strukturfelder aus Detail (für Bias-Audit)
    eintrag["arbeitgeberdarstellung"] = d.get("arbeitgeberdarstellung")
    eintrag["branchengruppe"] = d.get("branchengruppe")
    eintrag["branche"] = d.get("branche")
    eintrag["arbeitszeitmodelle"] = d.get("arbeitszeitmodelle")
    eintrag["befristung"] = d.get("befristung")
    time.sleep(sleep)
    return eintrag

In [ ]:
# Mehrere Suchanfragen für einen Mix — ändere/erweitere die Liste, falls dein Korpus zu einseitig wird.
# Ziel: ≥ 30 Anzeigen nach Dedup. Ausbildung + Festanstellung + verschiedene Berufsbezeichnungen mischen.
SUCHANFRAGEN = [
    {"was": "Fachinformatiker Daten- und Prozessanalyse", "size": 25},
    {"was": "Data Scientist", "size": 20},
    {"was": "Datenanalyst", "size": 20},
    {"was": "Business Intelligence", "size": 15},
    {"was": "Data Engineer", "size": 15},
]

treffer_roh: list[dict] = []
for q in SUCHANFRAGEN:
    resp = search(was=q["was"], size=q["size"])
    n_max = resp.get("maxErgebnisse", 0)
    treffer = extrahiere_treffer(resp, q["was"])
    print(f"  '{q['was']}': {len(treffer)} Treffer (von {n_max} verfügbar)")
    treffer_roh.extend(treffer)
    time.sleep(0.5)

print(f"\nGesamt vor Dedup: {len(treffer_roh)}")

# Dedup über refnr — eine Anzeige kann in mehreren Suchen auftauchen
gesehen: set[str] = set()
treffer_dedup: list[dict] = []
for t in treffer_roh:
    if not t["refnr"] or t["refnr"] in gesehen:
        continue
    gesehen.add(t["refnr"])
    treffer_dedup.append(t)

print(f"Nach Dedup: {len(treffer_dedup)}")
assert len(treffer_dedup) >= 30, f"Zu wenig Anzeigen ({len(treffer_dedup)}) — Suchanfragen erweitern."

In [ ]:
# Detail-Texte holen (mit Pause zwischen Requests, sonst rate-limited die API)
korpus: list[dict] = []
for i, t in enumerate(treffer_dedup, 1):
    angereichert = reichere_mit_text_an(t)
    korpus.append(angereichert)
    if i % 10 == 0:
        print(f"  {i}/{len(treffer_dedup)} Detail-Anzeigen geladen")

n_leer = sum(1 for k in korpus if not k.get("text"))
print(f"\nFertig: {len(korpus)} Anzeigen, davon {n_leer} ohne Beschreibungstext.")

# Anzeigen ohne Text fliegen raus — die taugen für Annotation nichts
korpus = [k for k in korpus if k.get("text")]
print(f"Nach Text-Filter: {len(korpus)} Anzeigen.")

In [ ]:
# JSONL schreiben (siehe CHEATSHEETS/jsonl.md)
with KORPUS_PATH.open("w", encoding="utf-8") as f:
    for eintrag in korpus:
        f.write(json.dumps(eintrag, ensure_ascii=False) + "\n")

print(f"{len(korpus)} Anzeigen geschrieben nach {KORPUS_PATH}")

### Block 1.2 — Korpus inspizieren

Vier Blicke aufs Material: Verteilungen, mehrfach genannte Firmen, Berufsbezeichnungen, Strukturfeld-Coverage. Output dieser Zellen ist die Faktenbasis für die Bias-Notiz unten.

In [ ]:
import pandas as pd

df = pd.read_json(KORPUS_PATH, lines=True)
print(f"Korpus-Größe: {len(df)} Anzeigen")
df.head(2)

In [ ]:
# Längen-Verteilung der Beschreibungstexte (Zeichen)
df["text_len"] = df["text"].str.len()
print(df["text_len"].describe().round(0))
df["text_len"].plot.hist(bins=20, title="Verteilung Textlängen (Zeichen)");

In [ ]:
# Firmen-Häufigkeit — wer taucht mehrfach auf?
firmen = df["firma"].value_counts()
print(f"Anzahl unterschiedlicher Firmen: {firmen.size}")
print("\nTop 10 Firmen:")
print(firmen.head(10))
print(f"\nFirmen mit ≥2 Anzeigen: {(firmen >= 2).sum()}")

In [ ]:
# Wie verteilen sich die Anzeigen über die Suchanfragen?
print("Anzeigen pro Suchanfrage (vor Dedup gezählt nach _suchbegriff):")
print(df["_suchbegriff"].value_counts())

In [ ]:
# Berufsbezeichnungen — was ist im Titel-Feld unterschiedlich?
print(f"Unterschiedliche Titel: {df['titel'].nunique()}")
print("\nTop 10 Titel:")
print(df["titel"].value_counts().head(10))

In [ ]:
# Regionale Verteilung — bundesweit oder klumpig?
print("Top 10 Orte:")
print(df["ort"].value_counts().head(10))
print("\nTop 10 Regionen:")
print(df["region"].value_counts().head(10))

In [ ]:
# Strukturfeld-Coverage: welche API-Felder sind wie oft befüllt?
strukturfelder = [
    "firma", "ort", "plz", "region", "eintrittsdatum",
    "aktuelleVeroeffentlichungsdatum", "externeUrl",
    "branche", "branchengruppe", "arbeitszeitmodelle", "befristung",
]
vorhanden = [c for c in strukturfelder if c in df.columns]
coverage = (df[vorhanden].notna() & (df[vorhanden] != "")).mean().round(2) * 100
print("Coverage der Strukturfelder (% befüllt):")
print(coverage.sort_values(ascending=False))

### Bias-Notiz (5–7 Sätze, eigene Worte)

> **Was hier rein soll:** was an *deinem* Korpus systematisch schief ist und was das für spätere Pipeline-Aussagen bedeutet. **Konkret werden — mit Zahlen oder Anzeigen-IDs aus den Zellen oben.** Nicht generisch ("könnte sein, dass …"), sondern: "X von Y meiner Anzeigen sind Z, also …".
>
> Anregungen aus dem Aufgabenblatt:
> - **Suchanfragen-Auswahl** — welche Begriffe hast du gewählt, welche nicht? Was übersiehst du dadurch?
> - **Plattform-Effekt** — was stellen Firmen *bei dieser API* überhaupt ein? (Bundesagentur ≠ LinkedIn ≠ StepStone)
> - **Region** — siehe Orts-Verteilung oben
> - **Vertragsart / Senioritätsmix** — sieht man im Titel-Feld
> - **Befüll-Lücken** — welche Strukturfelder hat die API gar nicht geliefert?

**Deine Bias-Notiz hier:**

_(5–7 Sätze, mit Zahlen aus den Zellen oben belegt)_

## Phase 2 — κ-Tabelle + drei Edge Cases